In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import numpy as np
from scipy.integrate import solve_ivp
from sympy import symbols, sympify
import torch

from halcyon.Models.MechanisticModel import ComplexODEModel, SBMLImportClassDummy
from halcyon.RegulatoryNetwork import RegulatoryNetwork
import libsbml
from sympy.parsing.sympy_parser import parse_expr

In [2]:
def test_import_from_sbml():
    sbml_path = "tests/genenetweaver_outputs/test_scalefree.xml"
    my_importclass_test = SBMLImportClassDummy.from_sbml(sbml_path)
    return my_importclass_test

In [3]:
imported = test_import_from_sbml()
print(imported)

None


In [4]:
alternative_max_name = "max_rate"

def __init__(self, regulatory_network: RegulatoryNetwork):
    super().__init__(regulatory_network)


def from_sbml(sbml_path: str):
    reader = libsbml.SBMLReader()
    document = reader.readSBML(sbml_path)  # type: ignore
    model = document.getModel()
    
    param_dict = dict()
    ODE_dict = dict()

    for i in range(model.getNumReactions()):
        reaction = model.getReaction(i)
        reaction_id = reaction.getId()

        expression, params = __parse_one_reaction(reaction)
        ODE_dict[reaction_id] = expression
        # Currently just get separate parameters for each reaction
        param_dict[reaction_id] = params
        
    flat_param_dict = {}
    for eqn, params in param_dict.items():
        for pname, pval in params.items():
            flat_param_dict[f"{eqn}_{pname}"] = pval
    
    # Replace parameter names in the expressions to match flat_param_dict keys
    for eqn, expr in ODE_dict.items():
        for pname in params.keys():
            flat_param_name = f"{eqn}_{pname}"
            expr = expr.subs(pname, flat_param_name)
        ODE_dict[eqn] = expr
        
    # combine synthesis and degradation reactions
    # Combine synthesis and degradation equations for each node
    combined_ODE_dict = {}
    print(set(rxn.rsplit("_", 1)[0] for rxn in ODE_dict.keys()))
    for node in set(rxn.rsplit("_", 1)[0] for rxn in ODE_dict.keys()):
        synth_key = f"{node}_synthesis"
        degr_key = f"{node}_degradation"
        synth_expr = ODE_dict.get(synth_key, 0)
        degr_expr = ODE_dict.get(degr_key, 0)
        combined_ODE_dict[node] = synth_expr - degr_expr
    ODE_dict = combined_ODE_dict
        
    return ODE_dict, flat_param_dict


def __parse_one_reaction(reaction: libsbml.Reaction):
    """_summary_

    Args:
        reaction (libsbml.Reaction): _description_

    Returns:
        Tuple of [Sympy expression, all_parameters]
    """
    kinetic_law = reaction.getKineticLaw()

    params = kinetic_law.getListOfParameters()
    param_dict = torch.nn.ParameterDict()
    for param in params:
        if param.getId().startswith("binds") or param.getId().startswith("num"):
            # Skip parameters that are not relevant for the model
            continue
        elif param.getId() == "max":
            # Rename max to max_rate to avoid conflicts with built-in max function
            param.setId(alternative_max_name)
        param_dict[param.getId()] = param.getValue()
    formula = kinetic_law.getFormula()
    # You can't keep the name max for the parameter because that throws an error in parsing here
    expression = parse_expr(formula.replace("max", alternative_max_name))
    
    return expression, param_dict



In [6]:
expression, param_dict = from_sbml("tests/genenetweaver_outputs/test_scalefree.xml")
print(type(expression["Node_0"]), expression["Node_0"]) 

{'Node_5', 'Node_1', 'Node_4', 'Node_6', 'Node_0', 'Node_7', 'Node_8', 'Node_3', 'Node_2', 'Node_9'}
<class 'sympy.core.add.Add'> -Node_0*Node_0_degradation_delta*cell + cell*max_rate*(a_0*(-(Node_2/k_4)**n_4*(Node_3/k_3)**n_3*(Node_4/k_1)**n_1*(Node_5/k_2)**n_2*((Node_2/k_4)**n_4 + 1)*((Node_3/k_3)**n_3 + 1)*((Node_5/k_2)**n_2 + 1)/((Node_4/k_1)**n_1 + 1) + 1) + a_1*(Node_2/k_4)**n_4*(Node_3/k_3)**n_3*(Node_4/k_1)**n_1*(Node_5/k_2)**n_2*((Node_2/k_4)**n_4 + 1)*((Node_3/k_3)**n_3 + 1)*((Node_5/k_2)**n_2 + 1)/((Node_4/k_1)**n_1 + 1))


In [7]:
param_dict

{'Node_0_synthesis_max_rate': 0.023730835370961087,
 'Node_0_synthesis_a_0': 1.0,
 'Node_0_synthesis_a_1': 0.04154790947364917,
 'Node_0_synthesis_k_1': 0.627290073109325,
 'Node_0_synthesis_k_2': 0.5362267186585814,
 'Node_0_synthesis_k_3': 0.19984182589221747,
 'Node_0_synthesis_k_4': 0.17613186082802715,
 'Node_0_synthesis_n_1': 3.211632278813405,
 'Node_0_synthesis_n_2': 1.0468915128905094,
 'Node_0_synthesis_n_3': 3.721183364937146,
 'Node_0_synthesis_n_4': 1.7444395476983998,
 'Node_0_degradation_delta': 0.023730835370961087,
 'Node_1_synthesis_max_rate': 0.02424460196266107,
 'Node_1_synthesis_a_0': 0.0014201704889018763,
 'Node_1_synthesis_a_1': 1.0,
 'Node_1_synthesis_k_1': 0.36031101578613745,
 'Node_1_synthesis_n_1': 1.3923741381380266,
 'Node_1_degradation_delta': 0.02424460196266107,
 'Node_2_synthesis_max_rate': 0.02052808819218263,
 'Node_2_synthesis_a_0': 0.5023119736779091,
 'Node_2_synthesis_a_1': 1.0,
 'Node_2_synthesis_a_2': 0.0,
 'Node_2_synthesis_a_3': 0.458770580

In [ ]:
# Assume state vector s is ordered as [Node_0, Node_1, ..., Node_9, cell]
# Map node names to indices in s for substitution
node_names = [f"Node_{i}" for i in range(10)]
node_indices = {name: idx for idx, name in enumerate(node_names)}

ODE_dict = {}

for rxn, expr in expression.items():
    # Prepare the list of variables for substitution
    def make_lambda(expr, param_dict):
        # sympy's lambdify expects a list of variables in order
        # Get all symbols in the expression
        symbols_in_expr = sorted(expr.free_symbols, key=lambda x: str(x))
        # Build a mapping from symbol name to index in s or parameter value
        # def func(s):
        #     subs = {}
        #     for sym in symbols_in_expr:
        #         name = str(sym)
        #         if name in node_indices:
        #             subs[sym] = s[node_indices[name]]
        #         elif name in params:
        #             subs[sym] = params[name]
        def func(s, param_dict):
            subs = {}
            for sym in symbols_in_expr:
                name = str(sym)
                # print(params.keys())
                if name in node_indices:
                    subs[sym] = s[node_indices[name]]
                elif name in param_dict:
                    subs[sym] = param_dict[name]
                else:
                    raise KeyError(f"Symbol {name} not found in state vector or parameters.")
            return float(expr.evalf(subs=subs))
        return func

    ODE_dict[rxn] = make_lambda(expr, param_dict)


ODE_dict["Node_0_synthesis"](np.zeros(11), param_dict)  # Test with a zero vector

0.0

In [ ]:
ODE_dict

{'Node_0_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_0_degradation': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_1_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_1_degradation': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_2_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_2_degradation': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_3_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_3_degradation': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_4_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_4_degradation': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_5_synthesis': <function __main__.make_lambda.<locals>.func(s, param_dict)>,
 'Node_5_degradation': <function __main__.make_lambda.<locals>.func(s, param_

In [ ]:
# replace each value in param dict with zero for testing
display(ODE_dict["Node_0_synthesis"](np.ones(11), param_dict)) 
param_dict = {k: 1.0 for k in param_dict.keys()}
display(ODE_dict["Node_0_synthesis"](np.ones(11), param_dict))  

-7504182.241489013

1.0

In [ ]:
import sympy

# 1) Define parameter dictionary
param_dict = {
    "pSPA": 1.0, "dSPA": 0.1, "assCOP1SPA1": 0.2, "dissCOP1SPA1": 0.05,
    "pCOP1": 1.0, "dCOP1": 0.1, "iCOP1_light": 0.1, "iCOP1_drought": 0.1,
    "iCOP1_heat_shoot": 1.0, "pCOP1_heat_shoot": 1.0, "iCOP1_SPA1_auto": 0.1,
    # Add any other missing parameters here
}

# 2) Define ODEs as lambdas that take the full state‐dict
ODE_dict = {
    "spa1_shoot": lambda s: (
        param_dict["pSPA"]
        - param_dict["dSPA"] * s["spa1_shoot"]
        - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
    ),
    "cop1_shoot": lambda s: (
        param_dict["pCOP1"] * param_dict["pCOP1_heat_shoot"]**s["heat"]
        - param_dict["dCOP1"] * s["cop1_shoot"]
        - (param_dict["iCOP1_light"]*s["light"] + param_dict["iCOP1_drought"]*s["drought"])
          * param_dict["iCOP1_heat_shoot"]**s["heat"] * s["cop1_shoot"]
        - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
    ),
    "cop1_spa1_shoot": lambda s: (
        param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        - param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
        - param_dict["iCOP1_SPA1_auto"] * s["cop1_spa1_shoot"]**2
    ),
    # … define hy5_shoot, hy5p_shoot, spa1_root, … etc. …
}

input_dict = {
    "light": lambda t: (t % 24 <= 16),  # light on for 16 hours
    "heat": lambda t: 0,                 # no heat stress
    "drought": lambda t: 0                # no drought stress
}

def combine_ODE_dict_and_param_dict(ODE_dict, param_dict):
    """
    Returns a new ODE_dict where each lambda takes a state dict and uses the correct param subdict.
    param_dict should be a dict of dicts, with keys matching ODE_dict.
    """
    combined_ODE_dict = {}
    for k, ode_fn in ODE_dict.items():
        params = param_dict.get(k, {})
        # Wrap the original lambda to inject the correct param subdict
        combined_ODE_dict[k] = (lambda ode_fn=ode_fn, params=params: (lambda s: ode_fn(s, params)))()
    return combined_ODE_dict

# Example usage:
# combined_ODE_dict = combine_ODE_dict_and_param_dict(ODE_dict, param_dict)


# 3) Get a list of the dependencies of each equation
# dependencies = {
#     "spa1_shoot":   ["cop1_shoot", "cop1_spa1_shoot"],
#     "cop1_shoot":   ["heat", "light", "drought", "spa1_shoot", "cop1_spa1_shoot"],
#     "cop1_spa1_shoot": ["cop1_shoot", "spa1_shoot"],
# }

def build_network_from_ODE_dict(ODE_dict: dict, input_dict: dict, remove_linear_self_loops: bool = True):
    dependencies = {}
    gene_names = list(ODE_dict.keys())
    input_names = list(input_dict.keys()) if input_dict is not None else ["light", "heat", "drought"]
    for var, ODE in ODE_dict.items():
        symbolic_vars = sympy.symbols(gene_names + input_names)
        # Build a dict for the lambda
        state_dict = {v: symbolic_vars[i] for i, v in enumerate(gene_names)}
        state_dict.update({name: symbolic_vars[len(gene_names) + j] for j, name in enumerate(input_names)})
        symbolic_expr = sympy.sympify(ODE(state_dict))
        # Find all dependencies
        deps = [
            str(sym) for sym in symbolic_expr.free_symbols
            if str(sym) in gene_names or str(sym) in input_names
        ]
        # Only add self-loop if dependence on self is higher order (nonlinear)
        if remove_linear_self_loops:
            if var in deps:
                var_sym = symbolic_vars[gene_names.index(var)]
                # Check if the highest power of var is > 1
                if sympy.degree(symbolic_expr, gen=var_sym) <= 1:
                    deps.remove(var)
        dependencies[var] = deps

    G = nx.DiGraph()
    for var, ODE in ODE_dict.items():
        G.add_node(var, ODE=ODE, type="network")
    
    for input_node, time_series_func in input_dict.items():
        G.add_node(input_node, time_series_func=time_series_func, type="input")
    
    for var, deps in dependencies.items():
        for d in deps:
            G.add_edge(d, var)
    return G

G = build_network_from_ODE_dict(ODE_dict, input_dict)



# 5) Write a wrapper to produce the derivative‐vector for scipy
def network_ODE(t, y):
    state = dict(zip(ODE_dict.keys(), y))
    # Update state with input values at time t
    for input_name, input_func in input_dict.items():
        state[input_name] = input_func(t)
    # Compute derivatives for each ODE variable
    dydt = np.zeros(len(ODE_dict))
    for i, (var, ode_func) in enumerate(ODE_dict.items()):
        dydt[i] = ode_func(state)
    return dydt


# 6) Integrate the ODEs
gene_names = list(ODE_dict.keys())
y0 = np.zeros(len(gene_names)) # initial conditions
ODEsolution = solve_ivp(network_ODE, [0, 7*24], y0, method="LSODA")


# create mechanistic model
# MM = ComplexMechanisticModel(RegulatoryNetwork(G), param_dict)
param_dict_torch = {k: torch.nn.Parameter(torch.tensor(v, dtype=torch.float32)) for k, v in param_dict.items()}
MM = ComplexODEMechanisticModel(ODE_dict, input_dict, param_dict_torch)
gene_names = list(ODE_dict.keys())

# integrate the ODEs
t_span = (0, 7*24)  # time span for integration
t_points = torch.linspace(t_span[0], t_span[1], 1000)  # time points for integration
# Ensure time points are properly spaced to avoid numerical issues
# Ensure initial conditions are non-negative and consistent with the equations
y0 = torch.zeros(len(gene_names))  # initial conditions (non-zero positive values)
# Add error handling to catch potential numerical issues during integration
# try:
#     solution = MM.integrate(y0, t_points)
# except Exception as e:
#     print(f"Integration failed: {e}")
#     solution = None
# print(MM.regulatory_network.nodes, G.nodes)
MMsolution = MM.integrate(y0, t_points)
# solution = solve_ivp(MM.forward, t_span=[0, 7*24], y0=y0, method="LSODA")

NameError: name 'ComplexODEMechanisticModel' is not defined